# MESOTES Pilot Analysis

This notebook loads the illustrative pilot JSONL files, inspects split distributions, and demonstrates the evaluation utilities on the shipped mock predictions.


In [ ]:
from pathlib import Path
import pandas as pd
from mesotes.loader import load_jsonl, load_records
from mesotes.metrics import evaluate_predictions
from mesotes.schema import PredictionRecord, ScenarioRecord

pilot_dir = Path('../data/pilot')
train = load_jsonl(pilot_dir / 'train.jsonl')
dev = load_jsonl(pilot_dir / 'dev.jsonl')
test = load_jsonl(pilot_dir / 'test_labels.jsonl')
all_records = pd.DataFrame(train + dev + test)
all_records[['id', 'split', 'domain', 'primary_sphere']].head()


In [ ]:
sphere_counts = all_records.groupby(['split', 'primary_sphere']).size().rename('count').reset_index()
domain_counts = all_records.groupby(['split', 'domain']).size().rename('count').reset_index()
sphere_counts
domain_counts


In [ ]:
gold_df = pd.json_normalize(all_records['gold'])
summary = pd.DataFrame({
    'needs_more_info_count': [int(gold_df['needs_more_info'].sum())],
    'no_mean_exception_count': [int(gold_df['no_mean_exception'].sum())],
    'false_midpoint_count': [int(gold_df['false_midpoint_action_id'].notna().sum())],
})
summary


In [ ]:
gold_records = load_records(pilot_dir / 'test_labels.jsonl', ScenarioRecord)
predictions = load_records(pilot_dir / 'mock_predictions.jsonl', PredictionRecord)
evaluate_predictions(predictions, gold_records)
